# MediaPipe Object Detection Learning

[![Open In Colab <](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShawnHymel/google-coral-micro-object-detection/blob/master/notebooks/mediapipe-object-detection-learning.ipynb)

```
Original authors: MediaPipeline (Google)
Modified by: Shawn Hymel
Date: December 16, 2023
```

Use transfer learning with Google MediaPipe to build a custom object detection model. Based on the example code from https://developers.google.com/mediapipe/solutions/customization/object_detector.

> **Note:** This script has been verified with TensorFlow v2.15.0.

To use this script, upload your dataset in [Pascal VOC format](http://host.robots.ox.ac.uk/pascal/VOC/) in an archive named *dataset.zip*. You can use a labeling tool like [labelImg](https://github.com/HumanSignal/labelImg) or [Make Sense](https://www.makesense.ai/) to create bounding box annotations in the Pascal VOC format.


Your data should be in the following format. Note that the directory names "Annotations" and "images" must be exactly as shown (with the capital 'A' and lowercase 'i').

```
dataset.zip
├── Annotations/
│   ├── image.01.xml
│   ├── image.02.xml
│   ├── ...
└── images/
    ├── image.01.jpg
    ├── image.02.jpg
    └── ...
```

Run through all the cells. Adjust the hyperparameters (`hparams`) as needed to achieve the desired accuracy. Ideally, you want your average precision (AP) to be greater than 90% to get a useful object detection model.

In [1]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Configuration

In [2]:
# Install MediaPipe and Edge TPU compiler
!python --version
!pip install --upgrade pip
!pip install mediapipe-model-maker
import os

# Settings
BASE_PATH = ".."  # go UP one directory to the parent folder

IMAGES_PATH = os.path.join(BASE_PATH, "images")

DATASET_ZIP_PATH = os.path.join(IMAGES_PATH, "dataset.zip")
DATASET_PATH = os.path.join(IMAGES_PATH, "dataset/")

TRAIN_SPLIT = 0.8

EXPORT_PATH = os.path.join(BASE_PATH, "exported_models/")
TFLITE_FLOAT32_NAME = "mymodel.tflite"
TFLITE_INT8_NAME = "mymodel_int8.tflite"

METADATA_PATH = os.path.join(EXPORT_PATH, "metadata.json")
METADATA_H_NAME = "metadata.hpp"
METADATA_H_PATH = os.path.join(EXPORT_PATH, METADATA_H_NAME)


Python 3.9.25


In [3]:
import os
import json
import tensorflow as tf

from mediapipe_model_maker import object_detector, quantization
files = os.listdir("..")
print(files)

2025-12-13 20:31:04.122556: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-13 20:31:04.189197: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-13 20:31:04.503374: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-13 20:31:04.503477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-13 20:31:04.560744: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/home/kate/Documents/MF2143/coral_env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.25) past its end of life. Google will update google.api_core with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
2025-12-13 20:31:05.770364: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/kate/Documents/MF2143/coral_env/lib/python3.9/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/home/kate/Documents/MF2143/coral_en

['reset.py', 'firmware', 'speech_comands.py', 'main.py', 'drive_to.py', '.git', 'notes.txt', 'turn.py', 'notebooks', 'send.zip', 'voice_commands.py', 'exported_models', 'states.txt', 'find_stuff.py', 'send', 'states.py', 'pick_up.py', '.vscode', 'tst.py', 'search_around.py', 'follow_line.py', 'search_for_line.py']


In [4]:
# Check TensorFlow version
print(tf.__version__)
assert tf.__version__.startswith('2')

2.15.1


In [5]:
# Settings
BASE_PATH = "."
DATASET_ZIP_PATH = os.path.join(BASE_PATH, "dataset.zip")
DATASET_PATH = os.path.join(BASE_PATH, "dataset/")
TRAIN_SPLIT = 0.8
EXPORT_PATH = os.path.join(BASE_PATH, "exported_models/")
TFLITE_FLOAT32_NAME = "model.tflite"
TFLITE_INT8_NAME = "model_int8.tflite"
METADATA_PATH = os.path.join(EXPORT_PATH, "metadata.json")
METADATA_H_NAME = "metadata.hpp"
METADATA_H_PATH = os.path.join(EXPORT_PATH, METADATA_H_NAME)

## Create dataset

Load and prepare the dataset for training and validation.

In [12]:
# Unzip dataset
!rm -rf {DATASET_PATH}
!unzip -q {DATASET_ZIP_PATH} -d {DATASET_PATH}

In [6]:
from mediapipe_model_maker import object_detector

# Load the dataset
data = object_detector.Dataset.from_pascal_voc_folder(DATASET_PATH)

# Split the dataset into separate training and validation sets
train_data, validation_data = data.split(TRAIN_SPLIT)

INFO:tensorflow:On image 0
INFO:tensorflow:On image 100


2025-12-13 20:31:20.343041: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-13 20:31:20.343752: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


INFO:tensorflow:On image 200
INFO:tensorflow:On image 300
INFO:tensorflow:On image 400
INFO:tensorflow:On image 500
INFO:tensorflow:On image 600
INFO:tensorflow:On image 700
INFO:tensorflow:On image 800
INFO:tensorflow:On image 900
INFO:tensorflow:On image 1000
INFO:tensorflow:On image 1100
INFO:tensorflow:On image 1200


## Train object detection model

Use transfer learning to retrain a model. Gather more/better data and adjust the hyperparameters (`hparams`) to ideally obtain a `total_loss` of less than 0.1 and an average precision (AP) of greater than 0.9.

In [7]:
# Load pre-trained model and specify hyperparameters
spec = object_detector.SupportedModels.MOBILENET_V2_I320
hparams = object_detector.HParams(
    learning_rate = 0.3,
    batch_size=8,
    epochs=50,
    export_dir=EXPORT_PATH,
)
options = object_detector.ObjectDetectorOptions(
    supported_model=spec,
    hparams=hparams,
)

In [8]:
# Retrain model
model = object_detector.ObjectDetector.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

/home/kate/Documents/MF2143/coral_env/lib/python3.9/site-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)


Using existing files at /tmp/model_maker/object_detector/mobilenetv2_i320
Model: "retina_net_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobile_net (MobileNet)      {'2': (None, 80, 80, 24   2257984   
                             ),                                  
                              '3': (None, 40, 40, 32             
                             ),                                  
                              '4': (None, 20, 20, 96             
                             ),                                  
                              '5': (None, 10, 10, 32             
                             0),                                 
                              '6': (None, 10, 10, 12             
                             80)}                                
                                                                 
 fpn (FPN)                   {'5': (None, 

INFO:tensorflow:Training the models...


Epoch 1/50


/home/kate/Documents/MF2143/coral_env/lib/python3.9/site-packages/keras/src/backend.py:452: UserWarning: `tf.keras.backend.set_learning_phase` is deprecated and will be removed after 2020-10-11. To update it, simply pass a True/False value to the `training` argument of the `__call__` method of your layer or model.
  warnings.warn(


129/129 [==============================] - 160s 1s/step - total_loss: 4.2616 - cls_loss: 3.9907 - box_loss: 0.0043 - model_loss: 4.2049 - val_total_loss: 1.0805 - val_cls_loss: 0.9099 - val_box_loss: 0.0023 - val_model_loss: 1.0237
Epoch 2/50
129/129 [==============================] - 142s 1s/step - total_loss: 0.8935 - cls_loss: 0.7051 - box_loss: 0.0026 - model_loss: 0.8367 - val_total_loss: 0.6390 - val_cls_loss: 0.5005 - val_box_loss: 0.0016 - val_model_loss: 0.5822
Epoch 3/50
129/129 [==============================] - 142s 1s/step - total_loss: 0.5566 - cls_loss: 0.3940 - box_loss: 0.0021 - model_loss: 0.4998 - val_total_loss: 0.4306 - val_cls_loss: 0.2966 - val_box_loss: 0.0015 - val_model_loss: 0.3739
Epoch 4/50
129/129 [==============================] - 142s 1s/step - total_loss: 0.4557 - cls_loss: 0.3025 - box_loss: 0.0019 - model_loss: 0.3988 - val_total_loss: 0.4008 - val_cls_loss: 0.2628 - val_box_loss: 0.0016 - val_model_loss: 0.3440
Epoch 5/50
129/129 [===================

In [22]:
# Evaluate model performance
loss, coco_metrics = model.evaluate(
    validation_data,
    batch_size=4,
)
print(f"Validation loss: {loss}")
print(f"Validation metrics: {coco_metrics}")

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.

## Export model

Save the model in three different formats:

 1. 32-bit floating point TensorFlow Lite (TFLite)
 2. 8-bit integer quantized TFLite
 3. TPU compiled and quantized TFLite|

Additionally, save the metadata (anchor box information) in a .h file that a resource-constrained device can recalculate the anchor boxes.



In [19]:
# Export 32-bit float model
model.export_model()

Exporting a floating point model


/home/kate/Documents/MF2143/coral_env/lib/python3.9/site-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)
/home/kate/Documents/MF2143/coral_env/lib/python3.9/site-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)


INFO:tensorflow:Assets written to: /tmp/tmp584y_k2k/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmp584y_k2k/saved_model/assets
2025-12-13 22:54:45.544053: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-12-13 22:54:45.544085: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-12-13 22:54:45.544288: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp584y_k2k/saved_model
2025-12-13 22:54:45.628735: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-12-13 22:54:45.628783: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp584y_k2k/saved_model
2025-12-13 22:54:45.880167: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-12-13 22:54:46.725891: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp584y_k2k/saved_model
2025-12-13 22:54:47.052294: I ten

NotFoundError: ./exported_models/model.tflite; No such file or directory

In [20]:
# Perform post-training quantization (8-bit integer) and save quantized model
quantization_config = quantization.QuantizationConfig.for_int8(
    representative_data=validation_data,
)
model.restore_float_ckpt()
model.export_model(
    model_name=TFLITE_INT8_NAME,
    quantization_config=quantization_config,
)

/home/kate/Documents/MF2143/coral_env/lib/python3.9/site-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)


Using existing files at /tmp/model_maker/object_detector/mobilenetv2_i320
Model: "retina_net_model_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobile_net_2 (MobileNet)    {'2': (None, 80, 80, 24   2257984   
                             ),                                  
                              '3': (None, 40, 40, 32             
                             ),                                  
                              '4': (None, 20, 20, 96             
                             ),                                  
                              '5': (None, 10, 10, 32             
                             0),                                 
                              '6': (None, 10, 10, 12             
                             80)}                                
                                                                 
 fpn_2 (FPN)                 {'5': (None

NotFoundError: Unsuccessful TensorSliceReader constructor: Failed to find any matching files for ./exported_models/float_ckpt

In [21]:
# Compile the model for Edge TPU
!edgetpu_compiler -s -o {EXPORT_PATH} {os.path.join(EXPORT_PATH, TFLITE_INT8_NAME)}

Edge TPU Compiler version 16.0.384591198
Error opening file for reading: ./exported_models/model_int8.tflite


In [ ]:
# Import model metadata
with open(METADATA_PATH, 'r') as file:
    metadata = json.load(file)

# Parse metadata
custom_metadata = metadata['subgraph_metadata'][0]['custom_metadata'][0]
anchors = custom_metadata['data']['ssd_anchors_options']['fixed_anchors_schema']['anchors']
num_values_per_keypoint = custom_metadata['data']['tensors_decoding_options']['num_values_per_keypoint']
apply_exponential_on_box_size = custom_metadata['data']['tensors_decoding_options']['apply_exponential_on_box_size']
x_scale = custom_metadata['data']['tensors_decoding_options']['x_scale']
y_scale = custom_metadata['data']['tensors_decoding_options']['y_scale']
w_scale = custom_metadata['data']['tensors_decoding_options']['w_scale']
h_scale = custom_metadata['data']['tensors_decoding_options']['h_scale']

In [ ]:
# Figure out when the resets (sectors) occur, the x/y increases, and width/height of anchors
reset_idxs = []
y_strides = []
x_strides = []
widths_per_section = []
widths = []
heights_per_section = []
heights = []
reset_flag = True
x_stride_flag = True
width_flag = True

# Go through all the anchors
num_anchors = len(anchors)
for i in range(num_anchors):

    # Store the first index
    if i == 0:
        reset_idxs.append(i)

    # Only measure strides on not 0 indexes
    else:

        # New section: reset flags
        if anchors[i]['y_center'] < anchors[i - 1]['y_center']:
            reset_idxs.append(i)
            reset_flag = True
            x_stride_flag = True
            width_flag = True

        # Measure Y increase (stride)
        if reset_flag:
            if anchors[i]['y_center'] > anchors[i - 1]['y_center']:
                y_inc = anchors[i]['y_center'] - anchors[i - 1]['y_center']
                y_strides.append(round(y_inc, 5))
                reset_flag = False

        # Measure X increase (stride)
        if x_stride_flag:
            if anchors[i]['x_center'] > anchors[i - 1]['x_center']:
                x_inc = anchors[i]['x_center'] - anchors[i - 1]['x_center']
                x_strides.append(round(x_inc, 5))
                x_stride_flag = False

    # Record widths and heights of the anchor boxes
    if width_flag:
        if i != 0 and anchors[i]['x_center'] > anchors[i - 1]['x_center']:
            widths.append(widths_per_section)
            widths_per_section = []
            heights.append(heights_per_section)
            heights_per_section = []
            width_flag = False
        else:
            width = anchors[i]['width']
            widths_per_section.append(round(width, 5))
            height = anchors[i]['height']
            heights_per_section.append(round(height, 5))

# Calculate the number of sectors
num_sectors = len(reset_idxs)

# Calculate the number of anchors per coordinate
num_anchors_per_coord = len(widths[0])

# Calculate the number of Xs in each Y
num_xs_per_y = []
for sector in range(num_sectors):
    num_xs_per_y.append(int(1.0 / x_strides[sector] * num_anchors_per_coord))

print(f"Number of anchors {num_anchors}")
print(f"Number of sectors: {num_sectors}")
print(f"Number of anchors per coordinate: {num_anchors_per_coord}")
print(f"Reset indexes: {reset_idxs}")
print(f"Number of Xs per Y: {num_xs_per_y}")
print(f"X strides: {x_strides}")
print(f"Y strides: {y_strides}")
print("Widths:")
for wps in widths:
    print(wps)
print("Heights:")
for hps in heights:
    print(hps)

In [ ]:
# Generate header file for metadata information
h_str = f"""\
// Filename: {METADATA_H_NAME}

#ifndef METADATA_HPP
#define METADATA_HPP

namespace metadata {{
    constexpr unsigned int num_anchors = {num_anchors};
    constexpr int apply_exp_scaling = {1 if apply_exponential_on_box_size else 0};
    constexpr float x_scale = {x_scale};
    constexpr float y_scale = {y_scale};
    constexpr float w_scale = {w_scale};
    constexpr float h_scale = {h_scale};
    constexpr unsigned int num_sectors = {num_sectors};
    constexpr unsigned int num_anchors_per_coord = {num_anchors_per_coord};
"""

# Print reset indexes
h_str += "    constexpr unsigned int reset_idxs[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{reset_idxs[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the number of X values for each Y value
h_str += "    constexpr unsigned int num_xs_per_y[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{num_xs_per_y[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the X strides
h_str += "    constexpr float x_strides[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{x_strides[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the Y strides
h_str += "    constexpr float y_strides[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{y_strides[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the anchor widths for each section
h_str += f"    constexpr float widths[{num_sectors}][{len(widths[0])}] = {{\r\n"
for i in range(num_sectors):
    h_str += "        {"
    for j in range(len(widths[0])):
        h_str += f"{widths[i][j]}"
        if j < len(widths[0]) - 1:
            h_str += ", "
    h_str += "}"
    if i < num_sectors - 1:
        h_str += ","
    h_str += "\r\n"
h_str += "    };\r\n"

# Print the anchor heights for each section
h_str += f"    constexpr float heights[{num_sectors}][{len(heights[0])}] = {{\r\n"
for i in range(num_sectors):
    h_str += "        {"
    for j in range(len(heights[0])):
        h_str += f"{heights[i][j]}"
        if j < len(heights[0]) - 1:
            h_str += ", "
    h_str += "}"
    if i < num_sectors - 1:
        h_str += ","
    h_str += "\r\n"
h_str += "    };\r\n"

# Close header file
h_str += """\
}

#endif // METADATA_HPP
"""

# write to .h file
with open(METADATA_H_PATH, 'w') as file:
    file.write(h_str)

In [ ]:
# Zip exported models
zip_name = os.path.normpath(EXPORT_PATH).split(os.sep)[-1] + ".zip"
zip_path = os.path.join(BASE_PATH, zip_name)
!zip -q -r {zip_path} {EXPORT_PATH}/*

In [ ]:
from IPython.display import FileLink
FileLink(zip_path)

In [ ]:
!zip -q -r exported_models.zip {os.path.join(EXPORT_PATH, "*")}